[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/02_data_validation.ipynb)

# 02 - Multi-Model Consensus Validation

**OMS Custom Embedding Model Pipeline - Step 2 of 6**

This notebook validates the synthetic (query, tool-description) pairs generated in Step 1
using multi-model consensus scoring. Each sampled pair is scored by 2+ LLM judges (Gemini + Claude)
for relevance on a 1-5 scale.

**Process:**
1. Load generated pairs from Step 1
2. Sample 500 pairs for validation
3. Score each pair with multiple LLMs
4. Compute inter-annotator agreement (Cohen's kappa)
5. Flag and review low-agreement pairs
6. Filter the full dataset based on validation results
7. Build final train/val/test splits (80/10/10)

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q google-generativeai anthropic openai scikit-learn datasets tqdm ipywidgets pyyaml

In [ ]:
import json
import os
import random
import re
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from tqdm.notebook import tqdm

print("All imports successful.")

In [ ]:
# Configure API keys (same as notebook 01)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print("Loaded API keys from Colab Secrets")
except (ImportError, Exception):
    print("Using environment variables for API keys.")

assert os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'), "GEMINI_API_KEY required"
assert os.getenv('ANTHROPIC_API_KEY'), "ANTHROPIC_API_KEY required for multi-model validation"
print("API keys verified.")

## 2. Load Generated Pairs

Load the JSONL output from Step 1.

In [ ]:
DATA_DIR = Path('data/raw')

# Find the JSONL file from Step 1
jsonl_files = list(DATA_DIR.glob('synthetic_queries_*.jsonl'))
if not jsonl_files:
    # Try Google Drive path
    DATA_DIR = Path('/content/drive/MyDrive/oms-models/data/raw')
    jsonl_files = list(DATA_DIR.glob('synthetic_queries_*.jsonl'))

assert len(jsonl_files) > 0, f"No JSONL files found in {DATA_DIR}. Run notebook 01 first."

# Load the most recent one
input_file = sorted(jsonl_files)[-1]
print(f"Loading: {input_file}")

records = []
with open(input_file, 'r') as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} (query, tool) pairs")
print(f"Unique tools: {len(set(r['tool_name'] for r in records))}")
print(f"Query types: {Counter(r['query_type'] for r in records).most_common(5)}")

## 3. Sample for Validation

Randomly sample 500 pairs for multi-model scoring. The sample is stratified by source type to ensure balanced coverage.

In [ ]:
VALIDATION_SAMPLE_SIZE = 500

random.seed(42)  # Reproducibility

# Stratified sampling by source
by_source = {}
for r in records:
    src = r.get('source', 'unknown')
    by_source.setdefault(src, []).append(r)

# Allocate proportionally
sample = []
for src, src_records in by_source.items():
    proportion = len(src_records) / len(records)
    n_sample = max(10, int(VALIDATION_SAMPLE_SIZE * proportion))
    n_sample = min(n_sample, len(src_records))
    sample.extend(random.sample(src_records, n_sample))

# Trim or pad to target size
if len(sample) > VALIDATION_SAMPLE_SIZE:
    sample = random.sample(sample, VALIDATION_SAMPLE_SIZE)
elif len(sample) < VALIDATION_SAMPLE_SIZE:
    remaining = [r for r in records if r not in sample]
    sample.extend(random.sample(remaining, VALIDATION_SAMPLE_SIZE - len(sample)))

print(f"Validation sample: {len(sample)} pairs")
sample_sources = Counter(r['source'] for r in sample)
for src, count in sample_sources.most_common():
    print(f"  {src}: {count} ({100*count/len(sample):.1f}%)")

## 4. Multi-Model Scoring

Send each pair to Gemini and Claude for relevance scoring on a 1-5 scale:
- **5**: Perfect match -- query clearly describes this exact tool
- **4**: Strong match -- query would find this tool in top results
- **3**: Moderate match -- tool is relevant but not the best fit
- **2**: Weak match -- tool is tangentially related
- **1**: No match -- query and tool are unrelated

In [ ]:
import requests

SCORING_PROMPT_TEMPLATE = """You are evaluating the quality of a (search query, tool description) pair
for training an embedding model for medical AI tool retrieval.

Rate the relevance of the query to the tool on a scale of 1-5:
  5 = Perfect match: the query clearly and specifically describes this tool's functionality
  4 = Strong match: the query would naturally lead to finding this tool
  3 = Moderate match: the tool is relevant but not the ideal result
  2 = Weak match: only tangentially related
  1 = No match: query and tool are unrelated

QUERY: {query}

TOOL: {tool_name}
DESCRIPTION: {tool_description}

Respond with ONLY a single integer (1-5) and nothing else."""


def score_with_gemini(query: str, tool_name: str, tool_desc: str) -> int:
    """Score a pair using Gemini API."""
    api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={api_key}"
    prompt = SCORING_PROMPT_TEMPLATE.format(query=query, tool_name=tool_name, tool_description=tool_desc)

    try:
        resp = requests.post(url, json={
            'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {'temperature': 0.0, 'maxOutputTokens': 10},
        }, timeout=30)
        resp.raise_for_status()
        text = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
        score = int(re.search(r'[1-5]', text).group())
        return score
    except Exception as e:
        print(f"Gemini error: {e}")
        return -1


def score_with_claude(query: str, tool_name: str, tool_desc: str) -> int:
    """Score a pair using Anthropic Claude API."""
    from anthropic import Anthropic
    client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
    prompt = SCORING_PROMPT_TEMPLATE.format(query=query, tool_name=tool_name, tool_description=tool_desc)

    try:
        message = client.messages.create(
            model='claude-haiku-4-20250514',
            max_tokens=10,
            messages=[{'role': 'user', 'content': prompt}],
        )
        text = message.content[0].text.strip()
        score = int(re.search(r'[1-5]', text).group())
        return score
    except Exception as e:
        print(f"Claude error: {e}")
        return -1


print("Scoring functions defined. Ready for multi-model validation.")

In [ ]:
# Run multi-model scoring on the validation sample
# This takes ~30 minutes for 500 pairs with rate limiting

scored_pairs = []
RATE_DELAY = 1.0  # seconds between API calls (adjust for rate limits)

for i, pair in enumerate(tqdm(sample, desc="Scoring pairs")):
    query = pair['query']
    tool_name = pair['tool_name']
    tool_desc = pair.get('tool_description', '')

    # Score with both models
    gemini_score = score_with_gemini(query, tool_name, tool_desc)
    time.sleep(RATE_DELAY)

    claude_score = score_with_claude(query, tool_name, tool_desc)
    time.sleep(RATE_DELAY)

    scored_pairs.append({
        **pair,
        'gemini_score': gemini_score,
        'claude_score': claude_score,
        'avg_score': (gemini_score + claude_score) / 2 if gemini_score > 0 and claude_score > 0 else -1,
        'score_diff': abs(gemini_score - claude_score) if gemini_score > 0 and claude_score > 0 else -1,
    })

    # Periodic progress
    if (i + 1) % 50 == 0:
        valid = [p for p in scored_pairs if p['avg_score'] > 0]
        if valid:
            avg = sum(p['avg_score'] for p in valid) / len(valid)
            print(f"  [{i+1}/{len(sample)}] Running avg score: {avg:.2f}")

print(f"\nScoring complete. {len(scored_pairs)} pairs scored.")

## 5. Agreement Analysis

Compute inter-annotator agreement metrics between Gemini and Claude scorers.

In [ ]:
# Filter to pairs where both models returned valid scores
valid_scored = [p for p in scored_pairs if p['gemini_score'] > 0 and p['claude_score'] > 0]
print(f"Valid scored pairs: {len(valid_scored)} / {len(scored_pairs)}")

gemini_scores = [p['gemini_score'] for p in valid_scored]
claude_scores = [p['claude_score'] for p in valid_scored]

# Cohen's Kappa (weighted for ordinal data)
kappa = cohen_kappa_score(gemini_scores, claude_scores, weights='quadratic')
print(f"\nCohen's Kappa (quadratic weighted): {kappa:.3f}")
if kappa >= 0.8:
    print("  Interpretation: Almost perfect agreement")
elif kappa >= 0.6:
    print("  Interpretation: Substantial agreement")
elif kappa >= 0.4:
    print("  Interpretation: Moderate agreement")
else:
    print("  Interpretation: Fair or poor agreement")

# Score distributions
print(f"\n--- Score Distribution ---")
print(f"{'Score':<8} {'Gemini':>8} {'Claude':>8}")
for score_val in range(1, 6):
    g_count = gemini_scores.count(score_val)
    c_count = claude_scores.count(score_val)
    print(f"  {score_val:<8} {g_count:>8} {c_count:>8}")

# Average scores
print(f"\nGemini mean: {np.mean(gemini_scores):.2f} (std: {np.std(gemini_scores):.2f})")
print(f"Claude mean: {np.mean(claude_scores):.2f} (std: {np.std(claude_scores):.2f})")

# Confusion matrix
cm = confusion_matrix(gemini_scores, claude_scores, labels=[1, 2, 3, 4, 5])
print(f"\nConfusion Matrix (Gemini vs Claude):")
print(f"       Claude: 1    2    3    4    5")
for i, row in enumerate(cm):
    print(f"Gemini {i+1}: {row}")

## 6. Flag Low-Agreement Pairs

Identify pairs where the two models disagree significantly (score difference >= 2).

In [ ]:
DISAGREEMENT_THRESHOLD = 2  # Flag pairs with score difference >= this

flagged = [p for p in valid_scored if p['score_diff'] >= DISAGREEMENT_THRESHOLD]
print(f"Flagged pairs (diff >= {DISAGREEMENT_THRESHOLD}): {len(flagged)} / {len(valid_scored)} ({100*len(flagged)/len(valid_scored):.1f}%)")

print(f"\n{'='*80}")
print(f"FLAGGED PAIRS FOR REVIEW")
print(f"{'='*80}")

for i, p in enumerate(flagged[:20], 1):
    print(f"\n--- Flagged #{i} ---")
    print(f"  Query:    {p['query']}")
    print(f"  Tool:     {p['tool_name']}")
    print(f"  Gemini:   {p['gemini_score']}  |  Claude: {p['claude_score']}  |  Diff: {p['score_diff']}")
    desc = p.get('tool_description', 'N/A')
    print(f"  Desc:     {desc[:100]}{'...' if len(desc) > 100 else ''}")

## 7. Manual Review UI

Interactive widget to approve or reject flagged pairs. Uses ipywidgets for an in-notebook review experience.

**Note**: This cell requires ipywidgets support. In Colab, it works natively. Locally, you may need `jupyter nbextension enable --py widgetsnbextension`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Review state
review_decisions = {}  # index -> 'approve' or 'reject'
current_idx = [0]

# UI elements
output_area = widgets.Output()
status_label = widgets.HTML(value="")

def show_pair(idx):
    """Display the current flagged pair for review."""
    with output_area:
        clear_output(wait=True)
        if idx >= len(flagged):
            print("All flagged pairs reviewed!")
            approved = sum(1 for v in review_decisions.values() if v == 'approve')
            rejected = sum(1 for v in review_decisions.values() if v == 'reject')
            print(f"Approved: {approved}, Rejected: {rejected}")
            return

        p = flagged[idx]
        decision = review_decisions.get(idx, 'pending')
        print(f"Pair {idx + 1} / {len(flagged)}  [Status: {decision}]")
        print(f"{'='*60}")
        print(f"Query:       {p['query']}")
        print(f"Tool:        {p['tool_name']}")
        print(f"Category:    {p.get('category', 'N/A')}")
        print(f"Description: {p.get('tool_description', 'N/A')[:200]}")
        print(f"\nGemini: {p['gemini_score']}  |  Claude: {p['claude_score']}")

def on_approve(b):
    review_decisions[current_idx[0]] = 'approve'
    current_idx[0] += 1
    show_pair(current_idx[0])

def on_reject(b):
    review_decisions[current_idx[0]] = 'reject'
    current_idx[0] += 1
    show_pair(current_idx[0])

def on_skip(b):
    current_idx[0] += 1
    show_pair(current_idx[0])

def on_prev(b):
    current_idx[0] = max(0, current_idx[0] - 1)
    show_pair(current_idx[0])

approve_btn = widgets.Button(description='Approve', button_style='success', icon='check')
reject_btn = widgets.Button(description='Reject', button_style='danger', icon='times')
skip_btn = widgets.Button(description='Skip', button_style='info', icon='forward')
prev_btn = widgets.Button(description='Previous', button_style='', icon='backward')

approve_btn.on_click(on_approve)
reject_btn.on_click(on_reject)
skip_btn.on_click(on_skip)
prev_btn.on_click(on_prev)

button_box = widgets.HBox([prev_btn, approve_btn, reject_btn, skip_btn])

if flagged:
    display(output_area)
    display(button_box)
    show_pair(0)
else:
    print("No flagged pairs to review. All pairs had good agreement.")

## 8. Filter Dataset

Remove low-quality pairs based on validation results. A pair is kept if:
- Average score >= 3.0 (moderate or better relevance), OR
- It was not in the validation sample (keep unscored pairs by default), OR
- It was manually approved in the review step

In [ ]:
MIN_AVG_SCORE = 3.0

# Build set of rejected queries from scoring
rejected_queries = set()
for p in valid_scored:
    if p['avg_score'] < MIN_AVG_SCORE:
        rejected_queries.add(p['query'])

# Also reject manually rejected pairs
for idx, decision in review_decisions.items():
    if decision == 'reject' and idx < len(flagged):
        rejected_queries.add(flagged[idx]['query'])

# Re-approve manually approved pairs
for idx, decision in review_decisions.items():
    if decision == 'approve' and idx < len(flagged):
        rejected_queries.discard(flagged[idx]['query'])

# Filter the full dataset
filtered_records = [r for r in records if r['query'] not in rejected_queries]

print(f"Original dataset:  {len(records)} pairs")
print(f"Rejected:          {len(records) - len(filtered_records)} pairs")
print(f"Filtered dataset:  {len(filtered_records)} pairs")
print(f"Retention rate:    {100 * len(filtered_records) / len(records):.1f}%")

## 9. Build Final Splits

Create train/val/test splits with an 80/10/10 ratio. The split is stratified by tool name to ensure
that all queries for a given tool appear in the same split (prevents data leakage).

In [ ]:
from datasets import Dataset, DatasetDict

random.seed(42)

# Group by tool name for stratified splitting
by_tool = {}
for r in filtered_records:
    tool = r['tool_name']
    by_tool.setdefault(tool, []).append(r)

tool_names = list(by_tool.keys())
random.shuffle(tool_names)

# Split tools into train/val/test groups
n_tools = len(tool_names)
n_val = max(1, int(0.1 * n_tools))
n_test = max(1, int(0.1 * n_tools))
n_train = n_tools - n_val - n_test

train_tools = tool_names[:n_train]
val_tools = tool_names[n_train:n_train + n_val]
test_tools = tool_names[n_train + n_val:]

train_records = [r for t in train_tools for r in by_tool[t]]
val_records = [r for t in val_tools for r in by_tool[t]]
test_records = [r for t in test_tools for r in by_tool[t]]

print(f"Split results (by tool, no leakage):")
print(f"  Train: {len(train_records):>6} pairs ({len(train_tools)} tools)")
print(f"  Val:   {len(val_records):>6} pairs ({len(val_tools)} tools)")
print(f"  Test:  {len(test_records):>6} pairs ({len(test_tools)} tools)")
print(f"  Total: {len(train_records) + len(val_records) + len(test_records):>6} pairs")

# Verify no overlap
train_set = set(r['query'] for r in train_records)
val_set = set(r['query'] for r in val_records)
test_set = set(r['query'] for r in test_records)
assert len(train_set & val_set) == 0, "Train/val overlap!"
assert len(train_set & test_set) == 0, "Train/test overlap!"
assert len(val_set & test_set) == 0, "Val/test overlap!"
print("No data leakage detected across splits.")

## 10. Save Processed Datasets

Save the train/val/test splits in HuggingFace Dataset format and JSONL.

In [ ]:
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save as HuggingFace DatasetDict
dataset_dict = DatasetDict({
    'train': Dataset.from_list(train_records),
    'validation': Dataset.from_list(val_records),
    'test': Dataset.from_list(test_records),
})

hf_path = PROCESSED_DIR / 'oms_training_pairs'
dataset_dict.save_to_disk(str(hf_path))
print(f"HuggingFace DatasetDict saved to: {hf_path}")

# Also save as JSONL for portability
for split_name, split_records in [('train', train_records), ('val', val_records), ('test', test_records)]:
    jsonl_path = PROCESSED_DIR / f'{split_name}.jsonl'
    with open(jsonl_path, 'w') as f:
        for r in split_records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f"  {split_name}.jsonl: {len(split_records)} records")

# Save validation scores for reference
scores_path = PROCESSED_DIR / 'validation_scores.jsonl'
with open(scores_path, 'w') as f:
    for p in scored_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print(f"  Validation scores saved to: {scores_path}")

print(f"\nDataset info:")
print(dataset_dict)

---

## Next Steps

Proceed to **03_train_embeddinggemma.ipynb** to fine-tune EmbeddingGemma-300M on the validated training pairs.

**Files produced by this notebook:**
- `data/processed/oms_training_pairs/` -- HuggingFace DatasetDict (train/val/test)
- `data/processed/train.jsonl` -- Training split
- `data/processed/val.jsonl` -- Validation split
- `data/processed/test.jsonl` -- Test split
- `data/processed/validation_scores.jsonl` -- Multi-model scoring results